# 13 可重現研究 — 參考解答

松柏護理之家退伍軍人症可重現分析流程練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## 題目 1：建立疫情摘要 dict

In [ ]:
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "n_residents": len(df),
    "n_infected": n_infected,
    "n_deaths": n_deaths,
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{n_deaths / n_infected:.1%}",
}

print("=== 疫情摘要 ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n→ 280 位住民、121 人感染、19 人死亡")
print("→ 侵襲率 43.2%、致死率 15.7%")

## 題目 2：可重現檢查清單

In [ ]:
checks = {
    "uv.lock 存在": Path("uv.lock").exists(),
    "pyproject.toml 存在": Path("pyproject.toml").exists(),
    "資料檔存在": Path("data/synthetic/legionella_outbreak.csv").exists(),
}

print("=== 可重現檢查清單 ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n→ {'全部通過！' if all_pass else '有項目未通過'}")

print("\n=== 每個項目為什麼重要 ===")
print("  uv.lock → 確保所有套件版本一致")
print("  pyproject.toml → 定義專案的套件需求")
print("  資料檔 → 沒有輸入就沒有輸出")

## 題目 3（挑戰題）：摘要輸出與驗證

In [ ]:
import json
import sys

# 存成 CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)
print("已存檔：data/processed/summary.csv")

# 重新讀取並驗證
reloaded = pd.read_csv(output_path / "summary.csv")
print(f"\n=== 驗證 ===")
print(f"原始 n_residents: {summary['n_residents']}")
print(f"重讀 n_residents: {reloaded['n_residents'].iloc[0]}")
print(f"一致: {summary['n_residents'] == reloaded['n_residents'].iloc[0]}")

# 版本資訊
print(f"\n=== 環境版本 ===")
print(f"  Python: {sys.version.split()[0]}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

print("\n=== 可能導致結果不同的因素 ===")
print("  1. 套件版本不同（例如 pandas 行為改變）")
print("  2. Python 版本不同")
print("  3. 資料檔被修改或遺失")
print("  4. 有使用亂數但未固定 seed")
print("  5. 作業系統差異（浮點運算精度）")
print("\n→ 用 uv.lock + git 可以解決前 3 個問題")

### 解讀

- **摘要 dict** 是最小可驗證單位——任何人跑都應該得到 280 住民、121 感染、19 死亡
- **檢查清單** 確保環境完整，缺少任何一項都可能導致無法重現
- **版本記錄** 是除錯的關鍵——如果結果不同，先比對版本
- **可重現三要素**：固定資料 + 版本控制程式碼 + 鎖定環境